# ISTAT TRAFFIC DATA ANALYSIS

First, let's import all the needed packages

In [ ]:
# import packages
import os
import requests
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


# set style for plots
sns.set_style("whitegrid")

### UNDERSTANDING THE DATA
First let's retrieve data from the `.csv` file obtained by fetching the API

In [ ]:
def get_api(save_path: str, url: str, header: dict = None) -> None:
    """
    Fetches data from the specified API endpoint and saves it to a CSV file.
    
    Parameters
    ----------
    url (str):                  The API endpoint URL.
    header (dict, optional):    Headers for the API request.
    """

    try:
        # GET request 
        print("Fetching data from the url...")
        r = requests.get(url, headers=header)
        
        # Check request status
        r.raise_for_status()

        filename = save_path
        with open(filename, "wb") as file:
            file.write(r.content)

    except requests.exceptions.HTTPError as err:
        print(f"HTTP error occurred: {err}")
    except Exception as err:
        print(f"An error occurred: {err}")

In [ ]:
data = "data/istat_traffic_accidents.csv"

url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
header = {'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}

if not os.path.exists(data):
    # run the function to retrieve the data from the API
    get_api(save_path=data, url=url, header=header)

# Creating pandas DataFrame
df_raw = pd.read_csv(data)

print(df_raw.info())
df_raw.sample(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATAFLOW          573552 non-null  object 
 1   FREQ              573552 non-null  object 
 2   REF_AREA          573552 non-null  int64  
 3   DATA_TYPE         573552 non-null  object 
 4   RESULT            573552 non-null  object 
 5   TIME_PERIOD       573552 non-null  int64  
 6   OBS_VALUE         573552 non-null  int64  
 7   OBS_STATUS        0 non-null       float64
 8   NOTE_DS           0 non-null       float64
 9   NOTE_REF_AREA     0 non-null       float64
 10  NOTE_DATA_TYPE    0 non-null       float64
 11  NOTE_RESULT       0 non-null       float64
 12  NOTE_TIME_PERIOD  0 non-null       float64
 13  BASE_PER          0 non-null       float64
 14  UNIT_MEAS         0 non-null       float64
 15  UNIT_MULT         0 non-null       float64
dtypes: float64(9), int64

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
486579,IT1:41_983(1.0),A,83023,KILLINJ,F,2007,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
137302,IT1:41_983(1.0),A,16050,KILLINJ,M,2017,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
437746,IT1:41_983(1.0),A,75002,KILLINJ,F,2014,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
294340,IT1:41_983(1.0),A,41050,ROADACC,9,2003,37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
324784,IT1:41_983(1.0),A,53010,ROADACC,9,2023,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
252330,IT1:41_983(1.0),A,28097,KILLINJ,M,2016,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
223428,IT1:41_983(1.0),A,24016,KILLINJ,M,2010,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
500673,IT1:41_983(1.0),A,87027,KILLINJ,M,2019,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
564833,IT1:41_983(1.0),A,106006,KILLINJ,M,2006,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23278,IT1:41_983(1.0),A,2019,KILLINJ,F,2017,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


At first glance, we can see that we only have information for the first seven columns. Meanwhile, the others have all NaN values, meaning we don't have information for them yet.  
Now, let's take a closer look at the columns with data, such as the time period range.

In [13]:
all_df_columns = df_raw.columns

# selecting only columns with no-null values
df = df_raw[all_df_columns[:7]]

# see all the unique values in each column
for col in df.columns:
    u = df[col].unique()
    print(f"Unique values in {col}: {len(u)}")
    print(u)
    print("---")

Unique values in DATAFLOW: 1
['IT1:41_983(1.0)']
---
Unique values in FREQ: 1
['A']
---
Unique values in REF_AREA: 8578
[  1001   1002   1003 ... 111105 111106 111107]
---
Unique values in DATA_TYPE: 2
['KILLINJ' 'ROADACC']
---
Unique values in RESULT: 3
['F' 'M' '9']
---
Unique values in TIME_PERIOD: 24
[2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013 2014
 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
---
Unique values in OBS_VALUE: 1830
[  10    7   13 ... 1081  880  969]
---


The first information we can gather is that our data range is 24 years, from 2001 to 2024. In the dataset, there are two types of accidents: "KILLINJ" and "ROADACC." From the tags, we can hypothesize that "KILLINJ" is a fatal or a serious accident and "ROADACC" is a simple road accident. However, we still have to understand what the other columns represent.  
- DATAFLOW: the possible code that represents the methodology of how the data are inserted into the dataset. Since we have only one unique value, it could explain why all the records were registered in the same way. This is also not useful for our analysis.
- FREQ: It could possibly represent the frequency, but we only have one unique value, "A." We have to investigate what this code means.
- REF_AREA: A code that represents the location or city where the incident occurred.
- RESULT: For this, we only have three distinct values: two letters and a number. More in depth information is required.
- OBS_VALUES: It could represent the total number of incidents that occurred in that specific area and time.
   

After searching the internet we found information about the structure and meaning of the dataset.  
From the official ISTAT website (https://www.istat.it/classificazioni-e-strumenti/web-services-sdmx/), we learned that the data in the APIs follows the SDMX standard protocol. Therefore, we were able to access the glossary for this structure on the official website, where we could download the universal glossary (https://sdmx.org/wp-content/uploads/SDMx_Glossary_version_2_1-Final-2.docx).   
Now that we have this information, we can describe each column of the data frame to help us continue the analysis.
- **DATAFLOW:** `41_983` represents the unique code that ISTAT uses to indicate the SDMX data flow *Incidenti, Morti e Feriti - Comuni* where the prefix *IT1* indicates Italy and *(1.0)* could indicate the version. Information obtained from the website https://datiistat.it/alldataflows#
- **FREQ:** rappresents the *"time interval at which observations occur over a given time period."* In this casa the letter A indicates a annual period (A = anno).
- **REF_AREA:** rappresents the *"Country or geographic area to which the measured statistical phenomenon relates"*. In this case each value indicates a city (comune)
- **DATA_TYPE:** This is not included in the glossary, but given the context, we can safely assume that "KILLINJ" represents a fatal or sirious accident and "ROADACC" represents a simple road accident.
- **RESULT:** no informtion found yet.
- **TIME_PERIOD:** rappresents the *"Timespan or point in time to which the observation actually refers"*. In this case each value indicates a the year
- **OBS_VALUE:** rappresents the *"Value of a particular variable"*. In this case, each value represents the total number of accidents that occurred in a specific city and year.
  
Since we don't have information for the `RESULT` entry only, let's investigate the data frame to understand its meaning.

In [20]:
print(df.loc[df["DATA_TYPE"] == "KILLINJ", "RESULT"].unique())
print(df.loc[df["DATA_TYPE"] == "ROADACC", "RESULT"].unique())

['F' 'M']
['9']


Here, we see how the three unique values in RESULT are specifically associated. The letters F and M are present only for the record with DATA_TYPE = KILLINJ, while the number 9 is present only when associated with DATA_TYPE = ROADACC.  
In the context of KILLINJ, where we mention how it represents cases of fatalities or injuries (the code is an aggregation of killed and injured), the letters F and M stand for FERITI and MORTI.  
After finding no information about the meaning of the number 9, we searched with AI tools and found that the number represents a specific code indicating "total." In simple terms, that record represents the "total" number of incidents, whether fatal or injury-related, without taking other information into account.  

So the updated legend of the dataset is the following:
- **DATAFLOW:** `41_983` represents the unique code that ISTAT uses to indicate the SDMX data flow *Incidenti, Morti e Feriti - Comuni* where the prefix *IT1* indicates Italy and *(1.0)* could indicate the version.
- **FREQ:** rappresents the *"time interval at which observations occur over a given time period."* In this casa the letter A indicates a annual period (A = anno).
- **REF_AREA:** rappresents the *"Country or geographic area to which the measured statistical phenomenon relates"*. In this case each value indicates a city (comune)
- **DATA_TYPE:** This is not included in the glossary, but given the context, we can safely assume that "KILLINJ" represents a fatal or sirious accident and "ROADACC" represents a simple road accident.
- **RESULT:** type of accident that occurred. It distinguishes between fatal (M) and injury (F), while "9" only represents the total number without any other information.
- **TIME_PERIOD:** rappresents the *"Timespan or point in time to which the observation actually refers"*. In this case each value indicates a the year
- **OBS_VALUE:** rappresents the *"Value of a particular variable"*. In this case, each value represents the total number of accidents that occurred in a specific city and year.

### COMPLETING THE DATA
From the INSTAT data, we only have the numeric code for the city (REF_AREA), but we don't know which city it indicates. To address this issue, we will use another SITUAS dataset that provides information about the numeric code, the corresponding city, and its population. This information is useful for calculating per capita statistical results.  
We downloaded the CSV file from https://situas.istat.it/web/#/territorio/body?id=74&dateFrom=2025-01-01, making sure to select January 1st of each year so that we have the population data for each previous year. This could be useful, as we could see how the proportion of accidents to the population changes over time.  

From these datasets, we only need information about the identification numeric code and the population size for each year. To accomplish this, we implemented a function that creates a smaller data frame containing all the necessary information.

In [45]:
def create_cities_df(path, features):

    year_list = np.arange(2002, 2026).astype('int32')

    df_temp_prev = pd.read_csv(path+str(year_list[0])+".csv", sep=";")
    df_temp_prev = df_temp_prev[features]

    for i in range(1, len(year_list)):
        df_temp = pd.read_csv(path+str(year_list[i])+".csv", sep=";")
        df_temp = df_temp[features]

        # concatenate the dataframes
        df_temp_prev = pd.concat([df_temp_prev, df_temp], ignore_index=True)
    
    # sorting by city code and year
    df_temp_prev = df_temp_prev.sort_values(by=['Codice Comune (numerico)', 'Anno (Popolazione residente)']).reset_index(drop=True)

    # save the new dataframe
    df_temp_prev.to_csv("data/cities_data.csv", index=False)


In [ ]:
cities_data = "data/situas/Comuni - Dimensione Data Indagine 01-01-"
cities_features = ["Codice Comune (numerico)", "Comune", "Popolazione residente", "Anno (Popolazione residente)"]

if not os.path.exists(cities_data):
    # run the function to retrieve the data from the API
    create_cities_df(path=cities_data, features=cities_features)

# Creating pandas DataFrame
cities_data_clean = "data/cities_data.csv"
df_cities = pd.read_csv(cities_data_clean)

print(df_cities.info())
df_cities.sample(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192577 entries, 0 to 192576
Data columns (total 4 columns):
 #   Column                        Non-Null Count   Dtype 
---  ------                        --------------   ----- 
 0   Codice Comune (numerico)      192577 non-null  int64 
 1   Comune                        192553 non-null  object
 2   Popolazione residente         192577 non-null  int64 
 3   Anno (Popolazione residente)  192577 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 5.9+ MB
None


,Codice Comune (numerico),Comune,Popolazione residente,Anno (Popolazione residente)
48088,16132,Mapello,5749,2003
106973,51009,Castelfranco di Sopra,2735,2001
183185,98010,Casalpusterlengo,15134,2018
108734,53009,Follonica,21409,2003
140147,69068,Pollutri,2311,2012
93215,35040,Scandiano,23396,2005
81746,27017,Fossò,7049,2017
143587,71023,Faeto,616,2020
56820,18043,Ceranova,2283,2021
173472,92039,Muravera,5271,2012


From the `.info()` we can see that there are some `NaN` values for the *Comune* columns. Let's investigate.

In [49]:
df_cities[df_cities["Comune"].isna()]

,Codice Comune (numerico),Comune,Popolazione residente,Anno (Popolazione residente)
3987,1168,NaN,7749,2001
3988,1168,NaN,7777,2002
3989,1168,NaN,7822,2003
3990,1168,NaN,7838,2004
3991,1168,NaN,7815,2005
3992,1168,NaN,7798,2006
3993,1168,NaN,7866,2007
3994,1168,NaN,7873,2008
3995,1168,NaN,7859,2009
3996,1168,NaN,7986,2010


In [ ]:
# Fast check to see if the city code are the same in both dataframes

print(df["REF_AREA"].unique(), len(df["REF_AREA"].unique()))
print(df_cities["Codice Comune (numerico)"].unique(), len(df_cities["Codice Comune (numerico)"].unique()))

[  1001   1002   1003 ... 111105 111106 111107] 8578
[  1001   1002   1003 ... 111105 111106 111107] 8578


For all the year the value of the city name is missing. So we can use any SITUAS dataset to check which city is missing

In [102]:
df_check = pd.read_csv("data/situas/Comuni - Dimensione Data Indagine 01-01-2025.csv", sep=";")

# filtering
df_check[df_check["Codice Comune (numerico)"] == 1168]

,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente)
164,1,1,1,201,1168,1168,NaN,NaN,TO,0,0,7830,2021,"24,6422",2025,7662,2024


The license plate number says "TO," so the city should be within the region of Turin.  
Let's take a look at all the cities in the Turin region that are mentioned in our dataset.

In [66]:
df_check.loc[df_check["Sigla automobilistica"] == "TO", "Comune"]

0                Agliè
1              Airasca
2         Ala di Stura
3      Albiano d'Ivrea
4               Almese
            ...       
307           Volpiano
308            Volvera
309            Mappano
310         Val di Chy
311          Valchiusa
Name: Comune, Length: 312, dtype: object

We should be able to find the missing city in our dataset using this website: https://www.comuni-italiani.it/001/lista.html. Let's scrape this page to get the data.

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By

def scrape_comuni():

    # Setup Chrome WebDriver
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument("--disable-gpu")
    driver = webdriver.Chrome(options=chrome_options)

    # Navigate to the page
    url = "https://www.comuni-italiani.it/001/lista.html"
    driver.get(url)

    # find the table by looking for one that contains the header "Comune", 
    rows = driver.find_elements(By.XPATH, "/html/body/span[3]/table[2]/tbody/tr[2]/td/table/tbody/tr")  # grab all of its table rows (<tr>)
    #"//table[.//td[contains(text(), 'Comune')]]//tr"
    
    data = []

    # Extract text
    for row in rows:
        cells = row.find_elements(By.TAG_NAME, "td")
        
        # The main data rows have exactly 5 columns
        if len(cells) == 5:
            row_data = [cell.text.strip() for cell in cells]
            
            # Skip the header row and the "Totale" row at the bottom
            if row_data[1] != "Comune" and row_data[0] != "Totale":
                data.append({
                    "Pos": row_data[0],
                    "Comune": row_data[1],
                    "Residenti": row_data[2],
                    "Densita_per_kmq": row_data[3],
                    "Numero_Famiglie": row_data[4]
                })
                
    driver.quit()
    
    return pd.DataFrame(data)



In [82]:
test = scrape_comuni()

print(test.info())
test

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 316 entries, 0 to 315
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Pos              316 non-null    object
 1   Comune           316 non-null    object
 2   Residenti        316 non-null    object
 3   Densita_per_kmq  316 non-null    object
 4   Numero_Famiglie  316 non-null    object
dtypes: object(5)
memory usage: 12.5+ KB
None


,Pos,Comune,Residenti,Densita_per_kmq,Numero_Famiglie
0,1,Torino,886.837,"6.812,9",438.954
1,2,Moncalieri,57.530,"1.207,9",27.400
2,3,Collegno,49.674,"2.741,4",22.278
3,4,Rivoli,48.798,"1.653,0",21.941
4,5,Nichelino,48.048,"2.327,9",20.682
...,...,...,...,...,...
311,312,Massello,53,"1,4",42
312,313,Ribordone,48,"1,1",42
313,314,Ingria,47,"3,2",29
314,315,Moncenisio,30,"7,5",26


It seems to work! Now, let's compare the two data frames to see which city is missing. We can simplify the research by filtering to select only a range of inhabitants where we expect to see the missing city.

In [91]:
# convert columns to numeric
test_cp = test.copy()

cols_to_convert = ['Residenti', 'Densita_per_kmq', 'Numero_Famiglie']
test_cp[cols_to_convert] = test_cp[cols_to_convert].replace({'\.': ''}, regex=True)
test_cp

,Pos,Comune,Residenti,Densita_per_kmq,Numero_Famiglie
0,1,Torino,886837,"6812,9",438954
1,2,Moncalieri,57530,"1207,9",27400
2,3,Collegno,49674,"2741,4",22278
3,4,Rivoli,48798,"1653,0",21941
4,5,Nichelino,48048,"2327,9",20682
...,...,...,...,...,...
311,312,Massello,53,"1,4",42
312,313,Ribordone,48,"1,1",42
313,314,Ingria,47,"3,2",29
314,315,Moncenisio,30,"7,5",26


In [92]:
test_cp[cols_to_convert] = test_cp[cols_to_convert].apply(pd.to_numeric, errors='coerce')
test_cp

,Pos,Comune,Residenti,Densita_per_kmq,Numero_Famiglie
0,1,Torino,886837,NaN,438954
1,2,Moncalieri,57530,NaN,27400
2,3,Collegno,49674,NaN,22278
3,4,Rivoli,48798,NaN,21941
4,5,Nichelino,48048,NaN,20682
...,...,...,...,...,...
311,312,Massello,53,NaN,42
312,313,Ribordone,48,NaN,42
313,314,Ingria,47,NaN,29
314,315,Moncenisio,30,NaN,26


In [101]:
test_cp[(test_cp['Residenti'] < 8000) & (test_cp['Residenti'] > 6000)]

,Pos,Comune,Residenti,Densita_per_kmq,Numero_Famiglie
44,45,Cumiana,7870,NaN,3559
45,46,Caluso,7492,NaN,3501
46,47,Luserna San Giovanni,7319,NaN,3500
47,48,Nole,6861,NaN,2910
48,49,Castiglione Torinese,6460,NaN,2724
49,50,Almese,6423,NaN,2935
50,51,Susa,6414,NaN,2773
51,52,Buttigliera Alta,6386,NaN,2766
52,53,Strambino,6251,NaN,2833
53,54,Cambiano,6086,NaN,2420


In [103]:
df_check[(df_check['Popolazione residente'] < 8000) & (df_check['Popolazione residente'] > 7000) & (df_check['Sigla automobilistica'] == 'TO')]

,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente)
45,1,1,1,201,1047,1047,Caluso,NaN,TO,0,0,7388,2021,"39,4924",2025,7309,2024
95,1,1,1,201,1097,1097,Cumiana,NaN,TO,0,0,7821,2021,"60,7335",2025,7793,2024
136,1,1,1,201,1139,1139,Luserna San Giovanni,NaN,TO,0,0,7162,2021,"17,7414",2025,7148,2024
164,1,1,1,201,1168,1168,NaN,NaN,TO,0,0,7830,2021,"24,6422",2025,7662,2024
309,1,1,1,201,1316,1316,Mappano,NaN,TO,0,0,7369,2021,"9,8174",2025,7270,2024
